# GPU Matrix-Multiplication Benchmark — Colab GPU runner

Builds and runs the `gpu-performance-benchmark` project on a **real NVIDIA GPU**:
CPU baseline, naive CUDA kernel, and shared-memory tiled CUDA kernel, with
correctness validation and generated plots.

**Requires a GPU runtime.** Before running: *Runtime → Change runtime type →*
*Hardware accelerator → GPU → Save.*

### Execution order (run every cell top to bottom)
1. **Check GPU & CUDA compiler** — prints GPU name, driver, and `nvcc` version; stops if no GPU.
2. **Get the source** — `git clone` your repo (set `REPO_URL`) or upload a zip.
3. **Install toolchain** — a recent CMake (so the build targets *this* GPU) plus pandas/matplotlib.
4. **Configure & build with CUDA** — fails loudly if CUDA is not actually enabled.
5. **CUDA correctness tests** — includes sizes that are *not* multiples of the tile (7, 15, 17, 31, 33, 100, 127, 129, 255, 257) and tile sizes 8/16/32.
6. **Run the benchmark** — CPU, naive CUDA, tiled CUDA → `results/benchmark_results.csv`.
7. **Generate plots & summary** → `plots/*.png`, `plots/summary.md`.
8. **Show results** inline.
9. **Package results** into a zip and download it.

Every step checks its exit code and raises on failure. No results are ever faked:
if a step fails, the notebook stops and shows the error.

## 1. Check NVIDIA GPU availability and CUDA compiler

In [ ]:
import subprocess, shutil, sys

def sh(cmd, cwd=None):
    """Run a command, stream its output, return the CompletedProcess."""
    print('$', ' '.join(cmd), flush=True)
    p = subprocess.run(cmd, cwd=cwd, text=True, stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT)
    print(p.stdout, end='' if p.stdout.endswith('\n') else '\n')
    return p

def check(cmd, cwd=None, what=None):
    """Run a command and raise (stopping the notebook) on non-zero exit."""
    p = sh(cmd, cwd=cwd)
    if p.returncode != 0:
        raise SystemExit(f'FAILED (exit {p.returncode}): {what or " ".join(cmd)}')
    return p

# 1a. GPU present?
if shutil.which('nvidia-smi') is None:
    raise SystemExit('No nvidia-smi on PATH. Enable a GPU runtime: '
                     'Runtime > Change runtime type > GPU, then rerun.')
if sh(['nvidia-smi']).returncode != 0:
    raise SystemExit('nvidia-smi failed: no GPU attached. Enable a GPU runtime and rerun.')

# 1b. GPU name, driver, memory, compute capability (best effort).
sh(['nvidia-smi', '--query-gpu=name,driver_version,memory.total,compute_cap',
    '--format=csv,noheader'])

# 1c. CUDA compiler version (required to build the kernels).
if shutil.which('nvcc') is None:
    raise SystemExit('nvcc (CUDA compiler) not found. A standard Colab GPU runtime '
                     'ships it under /usr/local/cuda/bin. Cannot build CUDA kernels.')
check(['nvcc', '--version'], what='nvcc --version')
print('\nGPU and CUDA compiler OK.')

## 2. Obtain the repository source

Set `REPO_URL` to your Git URL to clone (recommended once the project is on GitHub).
Leave it empty to upload a **zip of the project folder** instead.

In [ ]:
import os, glob, zipfile

REPO_URL = ''      # e.g. 'https://github.com/<you>/gpu-performance-benchmark.git'
WORK     = '/content/work'

if REPO_URL:
    check(['rm', '-rf', WORK])
    check(['git', 'clone', '--depth', '1', REPO_URL, WORK], what='git clone')
else:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Not on Colab and REPO_URL is empty: nothing to fetch.')
    print('REPO_URL is empty -> upload a .zip of the project folder.')
    check(['rm', '-rf', WORK]); os.makedirs(WORK, exist_ok=True)
    up = files.upload()
    zips = [n for n in up if n.lower().endswith('.zip')]
    if not zips:
        raise SystemExit('No .zip uploaded. Re-run this cell and upload a zip of the repo.')
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(WORK)

# Locate the project root (the directory that contains CMakeLists.txt).
hits = glob.glob(os.path.join(WORK, '**', 'CMakeLists.txt'), recursive=True)
if not hits:
    raise SystemExit(f'No CMakeLists.txt found under {WORK}. Is this the right source?')
PROJECT = os.path.dirname(sorted(hits, key=len)[0])
BUILD   = os.path.join(PROJECT, 'build')
print('PROJECT =', PROJECT)
assert os.path.isfile(os.path.join(PROJECT, 'scripts', 'run_benchmarks.py')), \
    'scripts/run_benchmarks.py missing - unexpected layout.'
check(['ls', '-la', PROJECT])

## 3. Install the build toolchain

Installs a recent **CMake** so that `CMAKE_CUDA_ARCHITECTURES=native` is used and
the kernels are compiled for *exactly* this runtime's GPU. pandas/matplotlib are
installed only if missing.

In [ ]:
import importlib.util

check([sys.executable, '-m', 'pip', 'install', '-q', 'cmake>=3.24'], what='pip install cmake')
for mod in ('pandas', 'matplotlib'):
    if importlib.util.find_spec(mod) is None:
        check([sys.executable, '-m', 'pip', 'install', '-q', mod], what=f'pip install {mod}')

# Use the freshly installed cmake (usually /usr/local/bin ahead on PATH).
CMAKE = shutil.which('cmake') or 'cmake'
check([CMAKE, '--version'], what='cmake --version')

## 4. Configure and build the project **with CUDA enabled**

If CUDA is not actually detected, this cell stops instead of silently producing a
CPU-only build.

In [ ]:
check([CMAKE, '-S', PROJECT, '-B', BUILD, '-DCMAKE_BUILD_TYPE=Release',
       '-DENABLE_CUDA=ON'], what='cmake configure')
check([CMAKE, '--build', BUILD, '-j'], what='cmake build')

# Confirm CUDA was really enabled (guards against a silent CPU-only build).
cache = open(os.path.join(BUILD, 'CMakeCache.txt')).read()
import re
m = re.search(r'CMAKE_CUDA_COMPILER:FILEPATH=(.+)', cache)
if not m or m.group(1).strip().upper() == 'NOTFOUND':
    raise SystemExit('CUDA was NOT enabled in the build. Aborting: this notebook is '
                     'for GPU benchmarking, not a CPU-only run.')
print('CUDA build confirmed:', m.group(1).strip())

## 5. CUDA correctness tests (including non-multiples of the tile size)

`ctest` runs the `correctness` suite on sizes 1, 2, 7, 15, 16, 17, 31, 32, 33, 64,
100, 127, 128, 129, 255, 256, 257 and the tiled kernel on tile sizes 8 / 16 / 32.
Every GPU result is compared element-wise against the CPU result.

In [ ]:
check(['ctest', '--test-dir', BUILD, '--output-on-failure'], what='ctest')

## 6. Run the benchmark pipeline (CPU, naive CUDA, tiled CUDA)

Runs the existing `scripts/run_benchmarks.py`: a correctness gate, then the
benchmark over each size, writing `results/benchmark_results.csv` and
`results/run_metadata.json`. Reduce `SIZES` if you are short on time (the 2048 CPU
run is the slowest part).

In [ ]:
SIZES = '128,256,512,1024,2048'   # e.g. '128,256,512,1024' to run faster
check([sys.executable, os.path.join(PROJECT, 'scripts', 'run_benchmarks.py'),
       '--sizes', SIZES], cwd=PROJECT, what='run_benchmarks.py')

## 7. Generate plots and the summary report

In [ ]:
check([sys.executable, os.path.join(PROJECT, 'scripts', 'plot_results.py')],
      cwd=PROJECT, what='plot_results.py')

## 8. Show the results

In [ ]:
import pandas as pd
from IPython.display import display, Image, Markdown

csv_path = os.path.join(PROJECT, 'results', 'benchmark_results.csv')
df = pd.read_csv(csv_path)
display(df[['impl','N','status','correct','time_ms_median','gflops',
            'speedup_vs_cpu','max_abs_err']])

display(Markdown(open(os.path.join(PROJECT, 'plots', 'summary.md')).read()))

for name in ['cpu_vs_gpu_time','naive_vs_tiled_kernel','speedup','gflops','transfer_overhead']:
    p = os.path.join(PROJECT, 'plots', name + '.png')
    if os.path.isfile(p):
        display(Image(p))

## 9. Package results & plots and download

Creates a timestamped zip of `results/` and `plots/` and downloads it to your
machine. Add those files to your repository (see the final cell).

In [ ]:
import datetime
stamp = datetime.datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
staging = f'/content/gpu_benchmark_output_{stamp}'
os.makedirs(staging, exist_ok=True)
for sub in ('results', 'plots'):
    src = os.path.join(PROJECT, sub)
    if os.path.isdir(src):
        shutil.copytree(src, os.path.join(staging, sub), dirs_exist_ok=True)
archive = shutil.make_archive(staging, 'zip', staging)
print('Wrote', archive)
try:
    from google.colab import files
    files.download(archive)
except Exception as e:
    print('Auto-download unavailable:', e)
    print('Download it manually from the Colab Files panel (folder icon on the left).')

## 10. Add the results to your repository

On your own machine, unzip the downloaded archive into the project folder so it
overwrites `results/` and `plots/`, then commit:

```bash
# from the project root, after unzipping the archive here
git add results/ plots/
git commit -m 'Add real GPU benchmark results from Colab'
git push
```

The `results/benchmark_results.csv` includes the device name, compute capability and
CUDA versions, so the numbers are traceable to the exact GPU that produced them.